# 1. Quick start: configuration and where everything lives

**What you will learn**
- How `path_config.json` becomes the paths `glmfitlins` uses (and how to check them).
- How FitLins options are chosen: command-line flag > `"fit"` block in the config > built-in default.
- Which OpenNeuro datasets already have FitLins derivatives, and where their files live on the public bucket.

Nothing here needs a cluster or a large download. Change `DATASET` and `TASK` below and re-run any cell.

In [1]:
import json, re, sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30, "display.width", 140)

# Make `nb_utils` importable whether Jupyter was started in notebooks/ or at the repo root
NB_DIR = Path.cwd() if (Path.cwd() / "nb_utils.py").exists() else Path.cwd() / "notebooks"
sys.path.insert(0, str(NB_DIR))
import nb_utils as nbu

REPO = NB_DIR.parent
SPECS = REPO / "statsmodel_specs"

from openneuro_glmfitlins.config import Config, ConfigError, load_config
from openneuro_glmfitlins.fit.fitlins import FIT_KEYS, FitOptions, build_fitlins_command, resolve_options

In [ ]:
DATASET = "ds003425"   # any OpenNeuro ID listed in section 4
TASK = "learning"      # a task of that dataset (section 4 lists them)

## 1. Your configuration

`glmfitlins config` prints the same thing. If you have no `path_config.json` yet, the cell falls back to a
placeholder so you can still follow along (copy `path_config.example.json` to `path_config.json` and edit it).

In [4]:
try:
    cfg = load_config()
    print(f"Loaded config from: {cfg.source}")
except ConfigError as err:
    print("No usable config found; using placeholder paths for this demo.\n")
    print(err)
    cfg = Config(datasets_folder=Path("/path/to/data"), tmp_folder=Path("/path/to/scratch"), repo_dir=REPO)

pd.Series({
    "datasets_folder": cfg.datasets_folder,
    "tmp_folder": cfg.tmp_folder,
    "repo_dir": cfg.repo_dir,
    "data_type": cfg.data_type,
    "fit block (path_config.json)": cfg.extra.get("fit"),
}, name="value").to_frame()

Loaded config from: /Users/demidenm/Desktop/Academia/Stanford/10_inkind/code/openneuro_glmfitlins/path_config.json


,value
datasets_folder,/Users/demidenm/Desktop/Academia/Stanford/10_i...
tmp_folder,/tmp/openneuro_fitlins
repo_dir,/Users/demidenm/Desktop/Academia/Stanford/10_i...
data_type,openneuro
fit block (path_config.json),"{'smoothing': '5:run:iso', 'estimator': 'nilea..."


## 2. Where files for one dataset go

`cfg.study(<id>)` returns every path for a dataset. Specs and group reports live **in this repo**; BIDS data,
fMRIPrep and FitLins outputs live under your `datasets_folder`.

In [5]:
paths = cfg.study(DATASET)
pd.Series({
    "spec dir (repo)": paths.spec_dir,
    "basic details": paths.basic_details,
    "model spec": paths.spec_file(TASK),
    "contrasts": paths.contrasts_file(TASK),
    "subjects": paths.subjects_file(TASK),
    "group report": paths.group_dir(TASK),
    "BIDS input": paths.bids_dir,
    "fMRIPrep derivatives": paths.fmriprep_dir,
    "FitLins output": paths.analysis_dir(TASK),
    "scratch / work dir": paths.scratch_dir(TASK),
}, name="path").to_frame()

,path
spec dir (repo),/Users/demidenm/Desktop/Academia/Stanford/10_i...
basic details,/Users/demidenm/Desktop/Academia/Stanford/10_i...
model spec,/Users/demidenm/Desktop/Academia/Stanford/10_i...
contrasts,/Users/demidenm/Desktop/Academia/Stanford/10_i...
subjects,/Users/demidenm/Desktop/Academia/Stanford/10_i...
group report,/Users/demidenm/Desktop/Academia/Stanford/10_i...
BIDS input,/Users/demidenm/Desktop/Academia/Stanford/10_i...
fMRIPrep derivatives,/Users/demidenm/Desktop/Academia/Stanford/10_i...
FitLins output,/Users/demidenm/Desktop/Academia/Stanford/10_i...
scratch / work dir,/tmp/openneuro_fitlins/fitlins/task-learning


## 3. How FitLins options are resolved

Three layers, first match wins: **CLI flag** > **`"fit"` block in `path_config.json`** > **built-in default**.
`resolve_options(cli, config_fit)` is the function `glmfitlins fit` uses. Below, the same function is called with
different inputs.

In [6]:
config_fit = cfg.extra.get("fit") or {}
cases = {
    "nothing set": ({}, None),
    "config block only": ({}, config_fit),
    "config + CLI flag (-s 8:run:iso)": ({"smoothing": "8:run:iso"}, config_fit),
}
rows = {}
for label, (cli, conf) in cases.items():
    opts = resolve_options(cli, conf)
    rows[label] = {k: getattr(opts, k) for k in FIT_KEYS}
pd.DataFrame(rows).T

,smoothing,estimator,n_cpus,mem_gb,space,desc_label,ignore,verbosity
nothing set,5:run:iso,nilearn,12,32,MNI152NLin2009cAsym,preproc,sub-.*_physio\.(json|tsv\.gz),3
config block only,5:run:iso,nilearn,12,32,MNI152NLin2009cAsym,preproc,sub-.*_physio\.(json|tsv\.gz),3
config + CLI flag (-s 8:run:iso),8:run:iso,nilearn,12,32,MNI152NLin2009cAsym,preproc,sub-.*_physio\.(json|tsv\.gz),3


`n_cpus` and `mem_gb` have no fixed default: they follow the SLURM request, or this machine when you are not on SLURM.

### Preview the FitLins command

This is what `glmfitlins fit -d <ds> -t <task> --dry-run` would run for the dataset above.

In [8]:
opts = resolve_options({"smoothing": "5:run:iso"}, config_fit)
cmd = build_fitlins_command(paths.bids_dir, paths.analysis_dir(TASK), paths.spec_file(TASK),
                            paths.fit_fmriprep_dir(), paths.scratch_dir(TASK), opts)
# one "flag value" pair per line
lines, i = [cmd[0]], 1
while i < len(cmd):
    if cmd[i].startswith("-") and i + 1 < len(cmd) and not cmd[i + 1].startswith("-"):
        lines.append(f"{cmd[i]} {cmd[i + 1]}"); i += 2
    else:
        lines.append(cmd[i]); i += 1
print(" \\ ".join(lines))

/Users/demidenm/Desktop/Academia/Stanford/10_inkind/code/openneuro_glmfitlins/.venv/bin/fitlins \ /Users/demidenm/Desktop/Academia/Stanford/10_inkind/test_data/input/ds003425 \ /Users/demidenm/Desktop/Academia/Stanford/10_inkind/test_data/analyses/ds003425/task-learning \ participant \ -m /Users/demidenm/Desktop/Academia/Stanford/10_inkind/code/openneuro_glmfitlins/statsmodel_specs/ds003425/ds003425-learning_specs.json \ -d /Users/demidenm/Desktop/Academia/Stanford/10_inkind/test_data/fmriprep/ds003425/derivatives \ --ignore sub-.*_physio\.(json|tsv\.gz) \ --drop-missing \ --space MNI152NLin2009cAsym \ --desc-label preproc \ --smoothing 5:run:iso \ --estimator nilearn \ --n-cpus 12 \ --mem-gb 32 \ -w /tmp/openneuro_fitlins/fitlins/task-learning \ -vvv


## 4. What exists on OpenNeuro right now

The notebooks read the public FitLins derivatives bucket over HTTPS (no AWS account needed):
`https://openneuro-derivatives.s3.amazonaws.com/fitlins/<dataset>-fitlins/task-<task>/`
(the same data as `aws s3 ls --no-sign-request s3://openneuro-derivatives/fitlins/`).

In [9]:
studies = nbu.list_studies()
local = sorted(p.name for p in SPECS.iterdir() if p.is_dir() and re.fullmatch(r"ds\d+", p.name))
print(f"{len(studies)} datasets with FitLins derivatives on OpenNeuro")
print(f"{len(local)} datasets with specs in this repository")
print("On OpenNeuro but no spec here:", sorted(set(studies) - set(local)) or "none")
print("Spec here but not (yet) on OpenNeuro:", sorted(set(local) - set(studies)) or "none")

54 datasets with FitLins derivatives on OpenNeuro
57 datasets with specs in this repository
On OpenNeuro but no spec here: none
Spec here but not (yet) on OpenNeuro: ['ds001233', 'ds001597', 'ds003397']


In [10]:
tasks = nbu.list_tasks(DATASET)
print(f"{DATASET} tasks on OpenNeuro: {tasks}")
assert TASK in tasks, f"{TASK!r} not found for {DATASET}; pick one of {tasks}"

print()
print("Derivative layout for this task:")
_dirs, _files = nbu.s3_list(nbu.DERIV_BUCKET, nbu.deriv_prefix(DATASET, TASK))
for d in _dirs:
    print("  ", d.replace(nbu.deriv_prefix(DATASET, TASK), ""))
for f in _files:
    print("  ", f["key"].replace(nbu.deriv_prefix(DATASET, TASK), ""))

ds003425 tasks on OpenNeuro: ['learning', 'prelearning', 'regulate', 'training']

Derivative layout for this task:
   node-dataLevel/
   node-runLevel/
   node-subjectLevel/
   reports/
   dataset_description.json


The three `node-*` folders match the three levels of the model (`runLevel`, `subjectLevel`, `dataLevel`).
`reports/` holds the FitLins HTML report. Notebook 4 explores these files in detail.

**Next:** [02_study_details_and_spec.ipynb](02_study_details_and_spec.ipynb)